# 03 · Fat tails & higher moments

Asset returns are **not** Gaussian: they have a taller central peak and far heavier tails than a normal of the same variance (Mandelbrot 1963, Fama 1965). sabia's **distribution** family — `kurt`, `skew`, `downside_dev`, `up_down_vol_ratio` — measures exactly these departures from normality.

The data is simulated with **Student-t innovations** (`_simulate.fat_tailed_ohlcv`, df=5), whose theoretical excess kurtosis is `6/(df-4)=6`. Requires `pip install plotly nbformat jupyterlab`.

In [1]:
import sys, pathlib, math
HERE = pathlib.Path.cwd()
sys.path.insert(0, str(HERE))
sys.path.insert(0, str(HERE.parent))

import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

pio.renderers.default = "plotly_mimetype+notebook_connected"
pio.templates.default = "plotly_white"

import sabia
import _simulate as sim
import _stats as st

SCHEMA = sim.default_schema()

# Student-t innovations (df=5): leptokurtic returns, fatter tails than a Gaussian of equal variance.
frame = sim.fat_tailed_ohlcv(n=750, df=5.0)
frame.tail(3)

timestamp,symbol,open,high,low,close,volume,vwap,dollar_volume,market_ret
"datetime[μs, UTC]",str,f64,f64,f64,f64,f64,f64,f64,f64
2023-01-18 00:00:00 UTC,"""FATT""",176.217407,176.67229,175.881506,176.375159,2.621085e6,176.309652,4.6229e8,0.010082
2023-01-19 00:00:00 UTC,"""FATT""",174.677463,175.613005,174.510693,175.366517,2.898812e6,175.163405,5.0835e8,-0.005878
2023-01-20 00:00:00 UTC,"""FATT""",176.956338,178.474886,176.289713,176.610555,3.221587e6,177.125052,5.6897e8,-0.004576


## Compute
`kurt` reports **excess** kurtosis (normal = 0). We also standardize returns to z-scores so the tail comparison against a normal is unit-free.

In [2]:
out = sabia.compute(
    frame,
    sabia.distribution.kurt(window=126),   # rolling EXCESS kurtosis
    sabia.distribution.skew(window=126),
    sabia.distribution.downside_dev(window=63),
    schema=SCHEMA,
)
df = frame.select("timestamp", "close").hstack(out).with_columns(
    (pl.col("close").log() - pl.col("close").log().shift(1)).alias("ret")
)
ret = df.get_column("ret").drop_nulls().to_numpy()
mu, sigma = float(ret.mean()), float(ret.std())
z = (ret - mu) / sigma
sample_excess_kurt = float((z**4).mean() - 3.0)
sample_skew = float((z**3).mean())
print(f"n={len(ret)}  excess kurtosis={sample_excess_kurt:.2f} (normal=0)  skew={sample_skew:.2f}")

n=749  excess kurtosis=2.69 (normal=0)  skew=0.21


## The distribution vs a normal
A fitted normal (same mean and variance) under- and over-shoots: it misses the peak and badly underestimates the tails (right panel, log scale).

In [3]:
xs = np.linspace(ret.min(), ret.max(), 400)
pdf = st.normal_pdf(xs, mu, sigma)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    "Return distribution vs normal — taller peak", "Same, log y — heavier tails"))
for c in (1, 2):
    fig.add_trace(go.Histogram(x=ret, histnorm="probability density", nbinsx=80,
                               marker_color="#1f77b4", opacity=0.55, name="returns",
                               showlegend=(c == 1)), row=1, col=c)
    fig.add_trace(go.Scatter(x=xs, y=pdf, name="normal", line=dict(color="#d62728", width=2),
                             showlegend=(c == 1)), row=1, col=c)
fig.update_yaxes(type="log", row=1, col=2)
fig.update_layout(height=420, barmode="overlay",
                  title=f"Leptokurtic returns (excess kurtosis {sample_excess_kurt:.1f}) — peaked centre, fat tails",
                  margin=dict(l=60, r=30, t=90, b=40))
fig.show()

## Tail risk, quantified
The sharpest way to feel fat tails: count how often returns breach k standard deviations versus what a normal permits. Beyond 3σ the gap is an order of magnitude — this is why Gaussian risk models understate crashes.

In [4]:
# Tail risk made concrete: how often do |z| > k standard deviations actually happen, vs what a
# normal predicts? Fat tails => the blue bars tower over the red ones at the extremes.
def normal_two_sided(k):
    return math.erfc(k / math.sqrt(2.0))  # P(|Z| > k) for a standard normal

ks = [1, 2, 3, 4, 5]
emp = [float(np.mean(np.abs(z) > k)) for k in ks]
nrm = [normal_two_sided(k) for k in ks]
labels = [f"&gt;{k}σ" for k in ks]
fig = go.Figure()
fig.add_trace(go.Bar(x=labels, y=emp, name="empirical", marker_color="#1f77b4",
                     text=[f"{p:.3%}" for p in emp], textposition="outside"))
fig.add_trace(go.Bar(x=labels, y=nrm, name="normal", marker_color="#d62728",
                     text=[f"{p:.3%}" for p in nrm], textposition="outside"))
fig.update_yaxes(type="log", title="P(|return| beyond k·σ)")
fig.update_layout(height=420, barmode="group",
                  title="Exceedance probabilities — extreme moves are far more common than a normal allows",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Higher moments through time
`kurt(126)` stays well above zero throughout; `skew(126)` wanders around zero (our t-noise is symmetric).

In [5]:
t = df.get_column("timestamp").to_list()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=("Rolling excess kurtosis, kurt(126) — persistently > 0",
                                    "Rolling skewness, skew(126)"))
fig.add_trace(go.Scatter(x=t, y=df.get_column("kurt_126").to_list(), line=dict(color="#6a3d9a", width=1.3),
                         showlegend=False), row=1, col=1)
fig.add_hline(y=0, row=1, col=1, line=dict(color="#455a64", width=1, dash="dash"))
fig.add_trace(go.Scatter(x=t, y=df.get_column("skew_126").to_list(), line=dict(color="#2ca02c", width=1.3),
                         showlegend=False), row=2, col=1)
fig.add_hline(y=0, row=2, col=1, line=dict(color="#455a64", width=1, dash="dash"))
fig.update_layout(height=520, title="Higher moments through time",
                  margin=dict(l=60, r=30, t=80, b=40))
fig.show()

## Takeaways

- Returns are **leptokurtic** — model tail risk with the empirical distribution, not a normal.
- `kurt` (excess) and `skew` turn 'fat tails' and 'asymmetry' into trailing, point-in-time features.
- Pair with `downside_dev` / `semivar_down` (notebook 02) to capture the *downside* tail that dominates drawdowns.